# linspace-out-param — ex1: use linspace(out=) to fill a pre-allocated buffer in place

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `linspace-out-param`. Running the final beacon cell reports progress against the `PyTorch: linspace out= param` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: linspace out= param` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linspace-out-param`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linspace-out-param"
DD_SUBTOPIC = "PyTorch: linspace out= param"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## PyTorch: `linspace(out=)` param — quick refresher

Most tensor-creation ops in PyTorch accept an `out=` kwarg that writes the result into a PRE-ALLOCATED tensor instead of allocating a fresh one:

```python
pre = torch.empty(11)             # allocate once, reuse forever
torch.linspace(0, 1, 11, out=pre)  # fills pre in place; returns pre
```

**Why this exists.** Inner loops that need a fresh buffer every iteration would otherwise pay the cost of allocation + deallocation N times. With `out=`, the loop allocates once and writes-through every iteration:

```python
buf = torch.empty(N)
for t_max in schedule:
    torch.linspace(0, t_max, N, out=buf)   # zero-alloc
    do_something_with(buf)
```

**Contract.** The `out` tensor must have the right shape and dtype (or PyTorch will resize it, which defeats the purpose). The function modifies `out` in place AND returns it — so you can chain: `y = torch.linspace(0, 1, 11, out=buf).pow(2)`.

**Same kwarg on every creation op.** `torch.zeros(... out=)`, `torch.arange(..., out=)`, `torch.randn(..., out=)`, `torch.empty_like(..., out=)`. Same semantics: in-place write, must be pre-sized.

**Don't confuse with `_inplace` (`fill_`, `zero_`).** Those are method-on-an-existing-tensor (`x.fill_(0)`). `out=` is a kwarg to a free function. Both achieve the same effect (modify in place) via different APIs.

### Exercise 1 — use linspace(out=) to fill a pre-allocated buffer in place

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.linspace(..., out=buf)` to fill a pre-allocated buffer in place across a zero-alloc inner loop, exploiting that `out=` shares storage with the buffer.
> Keywords: linspace, out, in-place, pre-allocation
> ```

**KCs targeted:** `out-kwarg-fills-in-place`, `out-tensor-aliasing`

Implement `ex1_fill_schedule(buf, t_maxes)`. The zero-alloc time-grid pattern used in diffusion samplers and physics solvers:

1. `buf` is a 1-D float tensor with `N = buf.numel()` elements. It is THE buffer — every call must write into it in place, never allocate a new tensor.
2. `t_maxes` is a list of floats; for each `t_max`, call `torch.linspace(0.0, t_max, N, out=buf)` to fill `buf` with the schedule `[0, t_max/(N-1), 2*t_max/(N-1), ..., t_max]`.
3. After each fill, append `buf.sum().item()` to a results list (so the test can verify the right value lived in `buf` at the right moment).
4. Return the results list.

Constraints:
- MUST use `out=buf` — do not allocate a new tensor per iteration.
- The returned tensor from `torch.linspace(..., out=buf)` is the SAME object as `buf` (aliased) — the test verifies this.

Inputs:
- `buf`: 1-D `Tensor` of any size.
- `t_maxes`: `list[float]`.

Output: `list[float]` (one per `t_max`).

In [ ]:
def ex1_fill_schedule(buf, t_maxes):
    N = buf.numel()
    results = []
    for t_max in t_maxes:
        t.linspace(0.0, t_max, N, out=buf)
        results.append(buf.sum().item())
    return results


<details><summary>Solution</summary>

```python
def ex1_fill_schedule(buf, t_maxes):
    N = buf.numel()
    results = []
    for t_max in t_maxes:
        t.linspace(0.0, t_max, N, out=buf)
        results.append(buf.sum().item())
    return results
```

**`out=buf` is the key.** The function fills `buf` in place AND returns it. We discard the return value and just read `buf.sum()` — same data, same storage.

**Why this matters for inner loops.** Diffusion samplers, Runge-Kutta solvers, and ODE integrators reuse a small time-grid buffer thousands of times per call. Without `out=`, each iteration allocates → fills → deallocates a fresh `N`-element tensor. With `out=`, the loop allocates ONCE.

**Same pattern, many ops.** `torch.zeros(out=)`, `torch.arange(out=)`, `torch.randn(out=)`, `torch.matmul(a, b, out=)`. Every op that takes `out=` follows the same in-place + return-aliased contract.

**Pre-sized buffer is the contract.** `buf` must be 1-D, the right size, and the right dtype, or PyTorch will silently resize it (defeating the no-alloc goal). In production code, allocate `buf` once at setup time and reuse it forever.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()